# Mammography denoising: reproducible companion
Muhammad Ahsan Ateeq | MSc Artificial Intelligence & Data Science

## Goal
Run the documented two-phase CDAE experiment with explicit patient-separated train, validation and test data. This refactor has **not** been trained on the original dataset. Archived scores belong to the historical experiment only.


## Setup
From the repository root, create a Python 3.11 environment and install `requirements.txt`. Select that environment as this notebook's kernel. Start with the README.


In [ ]:
from pathlib import Path
import os
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)
assert (root / "denoising").is_dir(), "Open from the repository root or notebooks directory"


## Data and assumptions
Obtain authorized, preprocessed 8-bit images yourself; no patient images are bundled. Copy `manifest.example.csv` to `manifest.csv`, then list your image paths, patient IDs and splits. Keep every patient in one split. Noise is Gaussian with sigma 0.4 on [0, 1] images after resizing; this differs from the original saved-noise pipeline.


In [ ]:
MANIFEST = "manifest.csv"
OUTPUT = "runs/notebook_experiment"
EVALUATION = "runs/notebook_evaluation"
PRETRAIN_EPOCHS = 1  # Small first run; historical configuration used 50
DENOISE_EPOCHS = 1  # Small first run; original code requested 100
SEED = 42
from denoising.data import read_manifest
splits = read_manifest(MANIFEST)
{k: len(v) for k, v in splits.items()}


## Training
This cell launches the portable CLI. Both phases select the best validation-loss checkpoint. The held-out test split is used only for final evaluation. Existing output folders are never overwritten.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "denoising.train", "--manifest", MANIFEST, "--output", OUTPUT, "--pretrain-epochs", str(PRETRAIN_EPOCHS), "--epochs", str(DENOISE_EPOCHS), "--seed", str(SEED)], check=True)


## Results
Compare the noisy-input baseline and reconstruction on the same images. Global PSNR and average per-image PSNR are different aggregations. Do not compare this small trial with the archived result as if they used the same inputs.


In [ ]:
subprocess.run([sys.executable, "-m", "denoising.evaluate", "--manifest", MANIFEST, "--model", f"{OUTPUT}/denoise.keras", "--output", EVALUATION], check=True)
import json
json.loads(Path(EVALUATION, "metrics.json").read_text())


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(Path(EVALUATION, "sample.png"))))


## Checks and next steps
Inspect the clean/noisy/reconstructed sample and per-image CSV, verify that denoising improves the baseline, and record dataset/preprocessing versions. Study multiple seeds and noise levels before drawing conclusions. These experiments do not establish clinical or diagnostic performance.

**Validation gap:** the companion notebook was not executed in the preparation environment because TensorFlow/Jupyter packages, dataset images and trained weights were unavailable. Run every cell in order after setup.
